# Statistical Analysis & Hypothesis Testing

## T-test: Even vs Odd

Do even numbers appear as often as odd numbers?
H0 (null): Even number mean frequency = Odd number mean frequency
H1 (alternate): Even number mean frequency is not equal odd number mean frequency
My prediction is that the mean frequencies won't be the same but they will be close enough to accept the null hypothesis.

In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

df = pd.read_csv("euromillions.csv")

nums = df.iloc[:,1:-1].values.flatten()

freq = pd.Series(nums).value_counts().sort_index()

evenfq = freq[freq.index % 2 == 0]
oddfq = freq[freq.index % 2 != 0]

t_stat, p_value = ttest_ind(evenfq, oddfq, equal_var=True)

print("Even mean:", evenfq.mean())
print("Odd mean:", oddfq.mean())
print("T-statistic:", t_stat)
print("P-value", p_value)

alp = 0.05

if p_value < alp:
    print("Reject H0: Significant difference")
else:
    print("Fail to reject H0: No significant difference")


Even mean: 265.84
Odd mean: 277.08
T-statistic: -0.2785406656970092
P-value 0.7817938257376165
Fail to reject H0: No significant difference


In [29]:
print(evenfq)
print(oddfq)

2     562
4     504
6     537
8     553
10    489
12    365
14    200
16    191
18    173
20    201
22    152
24    199
26    200
28    188
30    193
32    178
34    194
36    182
38    197
40    177
42    219
44    222
46    170
48    191
50    209
dtype: int64
1     517
3     571
5     542
7     542
9     540
11    452
13    202
15    199
17    211
19    216
21    212
23    218
25    202
27    205
29    215
31    182
33    170
35    202
37    205
39    191
41    178
43    179
45    201
47    180
49    195
dtype: int64


My prediction was correct, the p-value is significantly higher than 0.05 meaning we will fail to reject that even and odd numbers have similar frequencies.

## T-test: Even vs Odd by Column

Do even numbers appear as often as odd numbers in every column?
H0 (null): Even number mean frequency = Odd number mean frequency
H1 (alternate): Even number mean frequency is not equal odd number mean frequency
My prediction is that most columns will accept the null but there may be a column that rejects because of the randomness of lottery numbers.

In [27]:
num_cols = ['num_1', 'num_2', 'num_3', 'num_4', 'num_5', 'star_1', 'star_2']
results = []

for col in num_cols:
    freq = df[col].value_counts().sort_index()
    
    evenfqq = freq[freq.index % 2 == 0]
    oddfqq = freq[freq.index % 2 != 0]
    
    t_stat, p_value = ttest_ind(evenfqq, oddfqq, equal_var=False)
    
    null = "Reject H0" if p_value < alp else "Fail to Reject H0"
    
    results.append({'Column': col, 'Even Mean': evenfqq.mean(), 'Odd Mean': oddfqq.mean(), 't-stat': t_stat, 'p-value': p_value, 'Result': null})
    
    dfresults = pd.DataFrame(results)
    
print(dfresults)

   Column   Even Mean    Odd Mean    t-stat   p-value             Result
0   num_1   54.529412   56.222222 -0.087782  0.930580  Fail to Reject H0
1   num_2   42.090909   50.650000 -0.885142  0.381452  Fail to Reject H0
2   num_3   42.818182   45.318182 -0.303353  0.763132  Fail to Reject H0
3   num_4   47.400000   47.190476  0.022080  0.982498  Fail to Reject H0
4   num_5   58.166667   52.470588  0.284689  0.777677  Fail to Reject H0
5  star_1  176.400000  176.166667  0.003102  0.997593  Fail to Reject H0
6  star_2  162.333333  193.000000 -0.583736  0.574292  Fail to Reject H0


My prediction was partially correct, the p-value is significantly higher than 0.05 in every column, so every cplumn accepts the null hypothesis of even numbers and odd numbers being equal.

## ANOVA Test: Monday vs Wednesday vs Friday

Do the same numbers appear appear on different days (Mon, Wed, Fri)?
H0 (null): Mean values are equal across all days
H1 (alternate): At least one day has a different mean then the others
My predition is that the null will be accepted because the values will be too similair to reject.

In [55]:
from scipy.stats import f_oneway
df['date (dd-mm-yyyy)'] = pd.to_datetime(df['date (dd-mm-yyyy)'])
df['day'] = df['date (dd-mm-yyyy)'].dt.day_name()
   
dfdays = df[df['day'].isin(['Monday', 'Wednesday', 'Friday'])]
   
numbcols = df.columns[1:-2]
   
mon_value = dfdays[dfdays['day'] == 'Monday'][numbcols].values.flatten()
wed_value = dfdays[dfdays['day'] == 'Wednesday'][numbcols].values.flatten()
fri_value = dfdays[dfdays['day'] == 'Friday'][numbcols].values.flatten()

mon_mean = mon_value.mean()
wed_mean = wed_value.mean()
fri_mean = fri_value.mean()
   
f_stat, p_value = f_oneway(mon_value, wed_value, fri_value)
   
print("Monday Mean:", round(mon_mean, 3))
print("Wednesday Mean:", round(wed_mean, 3))
print("Friday Mean:", round(fri_mean, 3))
print("f-statistic:", round(f_stat, 3))
print("p-value:", round(p_value, 4))
   
if p_value < alp:
   print("Reject H0: At least one day is different")
else:
   print("Fail to Reject H0: No significant difference between days")

Monday Mean: 20.599
Wednesday Mean: 19.88
Friday Mean: 19.807
f-statistic: 0.874
p-value: 0.4173
Fail to Reject H0: No significant difference between days


My prediction was correct, all days are similar with Wednesday and Friday being extremely close and the p-value being higher than the alpha, the null fails to reject.

## ANOVA Test: Monday vs Wednesday vs Friday by Column

Do the same numbers appear appear on different days (Mon, Wed, Fri) in every column?
H0 (null): Mean values are equal across all days
H1 (alternate): At least one day has a different mean then the others
My predition is that the null will be accepted for most columns because the values will be too similair to reject, but there may be a column that rejects because of randomness.

In [51]:
results2 = []

for col in num_cols:
    mon = dfdays[dfdays['day'] == 'Monday'][col]
    wed = dfdays[dfdays['day'] == 'Wednesday'][col]
    fri = dfdays[dfdays['day'] == 'Friday'][col]
    
    mon_mean = mon.mean()
    wed_mean = wed.mean()
    fri_mean = fri.mean()
    
    f_stat, p_value = f_oneway(mon, wed, fri)
    
    results2.append({ "Column": col, "Mon Mean": round(mon_mean, 3), "Wed Mean": round(wed_mean, 3), "Fri Mean": round(fri_mean, 3), "f-stat": round(f_stat, 3), "p-value": round(p_value, 4), "Results": "Reject H0" if p_value < alp else "Fail to Reject H0"})
    
print(pd.DataFrame(results2))    

   Column  Mon Mean  Wed Mean  Fri Mean  f-stat  p-value            Results
0   num_1     8.359     8.167     8.236   0.025   0.9756  Fail to Reject H0
1   num_2    16.874    17.093    16.740   0.092   0.9125  Fail to Reject H0
2   num_3    26.340    25.315    25.392   0.561   0.5710  Fail to Reject H0
3   num_4    36.398    33.926    33.790   4.353   0.0131          Reject H0
4   num_5    44.117    42.343    42.559   2.620   0.0733  Fail to Reject H0
5  star_1     4.175     3.861     3.943   0.503   0.6049  Fail to Reject H0
6  star_2     7.932     8.454     7.990   1.784   0.1684  Fail to Reject H0


My prediction was correct, most columns accepted the null except for the num_4 column, meaning there's a significant difference between values across Monday, Wednesday, and Friday. This shows that at least one day is different than the others which is Monday because the average is 36 while the other days are around 33 and 34.